<a href="https://colab.research.google.com/github/rubendrt42-eng/SQL-Analitica-Ruben/blob/main/Etapa5_OT/Etapa5_ModeloFisico_OT_Equipo1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 5: Modelo dimensional físico y consultas de OT Consulting

**Equipo 1**, Soporte analítico para la toma de decisiones (TI2006B, Gpo 301), noviembre 2026

- Rubén David Rodríguez Treviño (A01198984)
- Erika Esquivel Correa (A00841206)
- Iván Fundora Velasco (A01198822)
- Julen Benages Díaz (A01832758)
- Javier Barrón Vargas (A00842507)

En la etapa 4 dibujamos la estrella. Aquí la construimos de verdad: creamos las tablas en SQL con sus llaves, las llenamos con los datos limpios y las usamos para contestar las 12 preguntas que nos planteamos desde la etapa 2. Al final sacamos el archivo que alimenta el tablero de Dirección.

## 1. Datos y limpieza

Usamos los mismos 11 CSV que sacamos de la app, con corte al 16 de noviembre. Si abres esto en Colab te va a pedir `datos_ot.zip`.

La limpieza no la copiamos aquí: el notebook baja el de la etapa 3 desde nuestro repo y lo corre tal cual, sin imprimir todo otra vez. Así, si algo se corrige allá, aquí también sale corregido. Al final revisamos que salgan los mismos números de la etapa 4.

In [1]:
import os, json, zipfile, sqlite3, contextlib, urllib.request, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 70)
try:
    from IPython.utils.capture import capture_output   # en Colab / Jupyter
except ImportError:
    capture_output = contextlib.nullcontext               # si se corre como script
if 'display' not in globals():
    display = print

CARPETA = 'datos_ot'
if not os.path.isdir(CARPETA):            # en Colab: subir datos_ot.zip
    from google.colab import files
    print('Sube el archivo datos_ot.zip')
    for nombre in files.upload():
        zipfile.ZipFile(nombre).extractall('.')

ETAPA3 = 'https://raw.githubusercontent.com/rubendrt42-eng/SQL-Analitica-Ruben/main/Etapa3_OT/Etapa3_Limpieza_OT_Equipo1.ipynb'
nb3 = json.loads(urllib.request.urlopen(ETAPA3).read())
with capture_output():                    # corre la etapa 3 completa, en silencio
    for celda in nb3['cells']:
        if celda['cell_type'] == 'code':
            fuente = ''.join(celda['source'])
            if 'CORTE' in globals():          # el script de actualización manda la fecha del día
                fuente = fuente.replace("pd.Timestamp('2026-11-16')", f"pd.Timestamp('{CORTE}')")
            exec(fuente, globals())
import matplotlib.pyplot as plt
plt.close('all')

personas = tabla[tabla['cuenta_persona'].eq(1)]
en_banca = personas[personas['estado_banca'].eq('En banca')]
print(f'Tabla acondicionada: {tabla.shape[0]} renglones x {tabla.shape[1]} columnas, {len(personas)} personas activas')
print(f"En banca: {len(en_banca)} personas que cuestan ${en_banca['costo_banca_mes'].sum():,.0f} al mes")

Tabla acondicionada: 76 renglones x 51 columnas, 75 personas activas
En banca: 22 personas que cuestan $1,363,346 al mes


## 2. Las tablas de la estrella

Repartimos las 51 columnas como quedó en la etapa 4: una tabla de hechos, `fact_asignacion`, y seis dimensiones. Cada dimensión tiene su llave `sk_` y guarda el código de la app como un dato más. Los que no tienen proyecto apuntan al registro 0 («Sin asignación») y las fechas que no existen, al -1 («Sin fecha»).

Agregamos dos tablas que no estaban en la estrella: `dim_habilidad` y un puente colaborador-habilidad. Las necesitamos para las preguntas 2 y 3, porque una persona tiene muchas habilidades y eso no cabe en un solo renglón de `dim_colaborador`. En Kimball a esto se le llama tabla puente (*bridge*).

In [2]:
def llave(df, columnas):
    # junta varias columnas en un texto para encontrar combinaciones iguales (los vacíos cuentan como '-')
    return df[columnas].apply(lambda r: '|'.join('-' if pd.isna(v) else str(v) for v in r), axis=1)

# dim_colaborador: una fila por persona activa
COLS_COLAB = ['id_colaborador', 'nombre', 'tipo_colaborador', 'es_facturable', 'puesto', 'nivel', 'departamento',
              'depto_origen', 'ciudad', 'nomina', 'antiguedad_anios', 'baja_programada', 'sueldo_total_integrado',
              'costo_faltante', 'costo_mensual', 'costo_hora', 'tarifa_min_usd_20', 'tarifa_min_usd_30',
              'n_habilidades', 'n_avanzado_experto', 'tecnologia_principal', 'captura_masiva', 'estado_banca']
dim_colaborador = personas[COLS_COLAB].sort_values('id_colaborador').reset_index(drop=True)
dim_colaborador.insert(0, 'sk_colaborador', range(1, len(dim_colaborador) + 1))

# dim_proyecto y dim_cliente: todos los de la app, más el registro 0
p = pro.sort_values('codigo_proyecto')
dim_proyecto = pd.concat([
    pd.DataFrame([{'sk_proyecto': 0, 'codigo_proyecto': 'P000', 'proyecto': 'Sin asignación', 'tipo_proyecto': 'No aplica'}]),
    pd.DataFrame({'sk_proyecto': range(1, len(p) + 1), 'codigo_proyecto': p['codigo_proyecto'].values,
                  'proyecto': p['nombre'].values, 'tipo_proyecto': p['tipo_proyecto'].values})], ignore_index=True)
c = cli.sort_values('codigo')
dim_cliente = pd.concat([
    pd.DataFrame([{'sk_cliente': 0, 'codigo_cliente': 'C0000', 'cliente': 'Sin cliente'}]),
    pd.DataFrame({'sk_cliente': range(1, len(c) + 1), 'codigo_cliente': c['codigo'].values, 'cliente': c['nombre'].values})],
    ignore_index=True)

# dim_fecha: calendario 2025-2028 y el -1
dias = pd.date_range('2025-01-01', '2028-12-31', freq='D')
MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
DIAS = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']
dim_fecha = pd.DataFrame({'sk_fecha': dias.strftime('%Y%m%d').astype(int), 'fecha': dias.strftime('%Y-%m-%d'),
                          'anio': dias.year, 'trimestre': dias.quarter, 'mes': dias.month,
                          'nombre_mes': [MESES[m - 1] for m in dias.month], 'dia': dias.day,
                          'dia_semana': [DIAS[d] for d in dias.dayofweek]})
dim_fecha = pd.concat([pd.DataFrame([{'sk_fecha': -1, 'fecha': None, 'anio': None, 'trimestre': None, 'mes': None,
                                      'nombre_mes': 'Sin fecha', 'dia': None, 'dia_semana': 'Sin fecha'}]), dim_fecha],
                      ignore_index=True)

# dim_detalle_asignacion: cada combinación distinta de rol y situación (dimensión basura de Kimball)
DET = ['rol_en_proyecto', 'rol_origen', 'es_indefinida', 'asignacion_vigente', 'ventana_liberacion', 'fuera_de_periodo']
combos = tabla[tabla['codigo_proyecto'].notna()][DET].drop_duplicates().sort_values(DET).reset_index(drop=True)
combos.insert(0, 'sk_detalle_asignacion', range(1, len(combos) + 1))
dim_detalle_asignacion = pd.concat([pd.DataFrame([{'sk_detalle_asignacion': 0, 'rol_en_proyecto': 'No aplica',
                                                   'rol_origen': 'No aplica', 'es_indefinida': None,
                                                   'asignacion_vigente': None, 'ventana_liberacion': 'No aplica',
                                                   'fuera_de_periodo': None}]), combos], ignore_index=True)

# dim_calidad: cada combinación de semáforo y campos que faltan
CAL = ['semaforo', 'campos_por_revisar']
dim_calidad = tabla[CAL].drop_duplicates().sort_values(CAL, na_position='first').reset_index(drop=True)
dim_calidad.insert(0, 'sk_calidad', range(1, len(dim_calidad) + 1))

# dim_habilidad y el puente (para las preguntas 2 y 3)
dim_habilidad = (hab.sort_values('codigo')[['codigo', 'descripcion', 'tecnologia', 'tipo', 'nombre_completo']]
                 .rename(columns={'codigo': 'codigo_habilidad'}).reset_index(drop=True))
dim_habilidad.insert(0, 'sk_habilidad', range(1, len(dim_habilidad) + 1))
puente_colaborador_habilidad = (mat.assign(id_colaborador=mat['uid_colaborador'].map(col.set_index('uid')['id']))
    .merge(dim_colaborador[['sk_colaborador', 'id_colaborador']], on='id_colaborador')
    .merge(dim_habilidad[['sk_habilidad', 'codigo_habilidad']], on='codigo_habilidad')
    [['sk_colaborador', 'sk_habilidad', 'nivel', 'nivel_num']].sort_values(['sk_colaborador', 'sk_habilidad'])
    .reset_index(drop=True))

# fact_asignacion: un renglón por persona activa en una asignación (76)
f = (tabla.merge(dim_colaborador[['sk_colaborador', 'id_colaborador']], on='id_colaborador', how='left')
          .merge(dim_proyecto[['sk_proyecto', 'codigo_proyecto']], on='codigo_proyecto', how='left')
          .merge(dim_cliente[['sk_cliente', 'codigo_cliente']], on='codigo_cliente', how='left'))
f[['sk_proyecto', 'sk_cliente']] = f[['sk_proyecto', 'sk_cliente']].fillna(0).astype(int)
a_sk = lambda s: pd.to_datetime(s).dt.strftime('%Y%m%d').astype(float).fillna(-1).astype(int)
f['sk_fecha_inicio'] = a_sk(f['inicio_asignacion'])
f['sk_fecha_fin'] = a_sk(f['fin_asignacion'])
f['sk_fecha_corte'] = int(FECHA_CORTE.strftime('%Y%m%d'))
f['sk_detalle_asignacion'] = llave(f, DET).map(dict(zip(llave(combos, DET), combos['sk_detalle_asignacion']))).fillna(0).astype(int)
f['sk_calidad'] = llave(f, CAL).map(dict(zip(llave(dim_calidad, CAL), dim_calidad['sk_calidad'])))
METRICAS = ['dedicacion', 'horas_mes', 'costo_asignacion_mes', 'ingreso_mes_40usd', 'ingreso_mes_45usd', 'ingreso_mes_50usd',
            'margen_40usd', 'margen_45usd', 'margen_50usd', 'costo_banca_mes', 'dias_para_liberarse', 'cuenta_persona',
            'puntaje_confianza']
fact_asignacion = f[['sk_colaborador', 'sk_proyecto', 'sk_cliente', 'sk_fecha_inicio', 'sk_fecha_fin', 'sk_fecha_corte',
                     'sk_detalle_asignacion', 'sk_calidad'] + METRICAS].copy()
fact_asignacion.insert(0, 'id_hecho', range(1, len(fact_asignacion) + 1))

TABLAS = {'dim_colaborador': dim_colaborador, 'dim_proyecto': dim_proyecto, 'dim_cliente': dim_cliente,
          'dim_fecha': dim_fecha, 'dim_detalle_asignacion': dim_detalle_asignacion, 'dim_calidad': dim_calidad,
          'dim_habilidad': dim_habilidad, 'fact_asignacion': fact_asignacion,
          'puente_colaborador_habilidad': puente_colaborador_habilidad}
pd.DataFrame([{'tabla': k, 'renglones': len(v), 'columnas': v.shape[1]} for k, v in TABLAS.items()])

,tabla,renglones,columnas
0,dim_colaborador,75,24
1,dim_proyecto,13,4
2,dim_cliente,9,3
3,dim_fecha,1462,8
4,dim_detalle_asignacion,16,7
5,dim_calidad,18,3
6,dim_habilidad,71,6
7,fact_asignacion,76,22
8,puente_colaborador_habilidad,1276,4


Salen los mismos tamaños que reportamos en la etapa 4: 75 colaboradores, 13 proyectos, 9 clientes, 1,462 fechas, 16 detalles, 18 combinaciones de calidad y 76 hechos con 22 columnas. El puente tiene 1,276 renglones, que son las habilidades de las personas activas.

## 3. Modelo físico en SQL

El modelo lógico dice qué tablas hay. El físico dice cómo se guardan: el tipo de cada columna, la llave primaria, las llaves foráneas y reglas (`CHECK`) para que no entre basura, por ejemplo un semáforo que no sea Verde, Amarillo o Rojo, o un nivel de habilidad fuera de 1 a 4.

Lo hicimos en SQLite porque corre en Colab sin instalar nada. El SQL es estándar, así que se pasa a BigQuery o PostgreSQL cambiando los tipos. El mismo script queda en `modelo_fisico.sql`.

In [3]:
DDL = """
-- Modelo dimensional físico · OT Consulting · Equipo 1 (TI2006B Gpo 301)
-- Estrella de la Etapa 4 (fact_asignacion + 6 dimensiones) + puente de habilidades.
-- Escrito en SQL estándar; corre igual en SQLite, PostgreSQL o BigQuery cambiando los tipos.

CREATE TABLE dim_colaborador (
    sk_colaborador          INTEGER PRIMARY KEY,
    id_colaborador          TEXT    NOT NULL UNIQUE,          -- llave natural de la app (OTC###)
    nombre                  TEXT    NOT NULL,
    tipo_colaborador        TEXT    NOT NULL,
    es_facturable           INTEGER NOT NULL CHECK (es_facturable IN (0, 1)),
    puesto                  TEXT    NOT NULL,
    nivel                   TEXT,
    departamento            TEXT,
    depto_origen            TEXT    NOT NULL,
    ciudad                  TEXT,
    nomina                  TEXT,
    antiguedad_anios        REAL,
    baja_programada         INTEGER NOT NULL CHECK (baja_programada IN (0, 1)),
    sueldo_total_integrado  REAL,
    costo_faltante          INTEGER NOT NULL CHECK (costo_faltante IN (0, 1)),
    costo_mensual           REAL,
    costo_hora              REAL,
    tarifa_min_usd_20       REAL,
    tarifa_min_usd_30       REAL,
    n_habilidades           INTEGER NOT NULL,
    n_avanzado_experto      INTEGER NOT NULL,
    tecnologia_principal    TEXT,
    captura_masiva          INTEGER NOT NULL CHECK (captura_masiva IN (0, 1)),
    estado_banca            TEXT    NOT NULL
        CHECK (estado_banca IN ('Asignado', 'En banca', 'Por confirmar', 'No facturable'))
);

CREATE TABLE dim_proyecto (
    sk_proyecto      INTEGER PRIMARY KEY,                     -- 0 = Sin asignación
    codigo_proyecto  TEXT NOT NULL UNIQUE,
    proyecto         TEXT NOT NULL,
    tipo_proyecto    TEXT NOT NULL
);

CREATE TABLE dim_cliente (
    sk_cliente      INTEGER PRIMARY KEY,                      -- 0 = Sin cliente
    codigo_cliente  TEXT NOT NULL UNIQUE,
    cliente         TEXT NOT NULL
);

CREATE TABLE dim_fecha (
    sk_fecha    INTEGER PRIMARY KEY,                          -- AAAAMMDD; -1 = Sin fecha
    fecha       TEXT,
    anio        INTEGER,
    trimestre   INTEGER CHECK (trimestre BETWEEN 1 AND 4),
    mes         INTEGER CHECK (mes BETWEEN 1 AND 12),
    nombre_mes  TEXT NOT NULL,
    dia         INTEGER,
    dia_semana  TEXT NOT NULL
);

CREATE TABLE dim_detalle_asignacion (
    sk_detalle_asignacion  INTEGER PRIMARY KEY,               -- 0 = No aplica
    rol_en_proyecto        TEXT,                              -- vacío = la app no tiene rol ni puesto
    rol_origen             TEXT NOT NULL,
    es_indefinida          INTEGER CHECK (es_indefinida IN (0, 1)),
    asignacion_vigente     INTEGER CHECK (asignacion_vigente IN (0, 1)),
    ventana_liberacion     TEXT NOT NULL,
    fuera_de_periodo       INTEGER CHECK (fuera_de_periodo IN (0, 1))
);

CREATE TABLE dim_calidad (
    sk_calidad          INTEGER PRIMARY KEY,
    semaforo            TEXT NOT NULL CHECK (semaforo IN ('Verde', 'Amarillo', 'Rojo')),
    campos_por_revisar  TEXT
);

CREATE TABLE dim_habilidad (
    sk_habilidad      INTEGER PRIMARY KEY,
    codigo_habilidad  TEXT NOT NULL UNIQUE,
    descripcion       TEXT NOT NULL,
    tecnologia        TEXT NOT NULL,
    tipo              TEXT NOT NULL,
    nombre_completo   TEXT NOT NULL
);

CREATE TABLE fact_asignacion (
    id_hecho               INTEGER PRIMARY KEY,
    sk_colaborador         INTEGER NOT NULL REFERENCES dim_colaborador (sk_colaborador),
    sk_proyecto            INTEGER NOT NULL REFERENCES dim_proyecto (sk_proyecto),
    sk_cliente             INTEGER NOT NULL REFERENCES dim_cliente (sk_cliente),
    sk_fecha_inicio        INTEGER NOT NULL REFERENCES dim_fecha (sk_fecha),
    sk_fecha_fin           INTEGER NOT NULL REFERENCES dim_fecha (sk_fecha),
    sk_fecha_corte         INTEGER NOT NULL REFERENCES dim_fecha (sk_fecha),
    sk_detalle_asignacion  INTEGER NOT NULL REFERENCES dim_detalle_asignacion (sk_detalle_asignacion),
    sk_calidad             INTEGER NOT NULL REFERENCES dim_calidad (sk_calidad),
    dedicacion             REAL,
    horas_mes              REAL,
    costo_asignacion_mes   REAL,
    ingreso_mes_40usd      REAL,
    ingreso_mes_45usd      REAL,
    ingreso_mes_50usd      REAL,
    margen_40usd           REAL,
    margen_45usd           REAL,
    margen_50usd           REAL,
    costo_banca_mes        REAL,
    dias_para_liberarse    INTEGER,
    cuenta_persona         INTEGER NOT NULL CHECK (cuenta_persona IN (0, 1)),
    puntaje_confianza      REAL    NOT NULL CHECK (puntaje_confianza BETWEEN 0 AND 1)
);

-- Puente (bridge) de Kimball: una persona tiene muchas habilidades y una habilidad la tienen muchas personas
CREATE TABLE puente_colaborador_habilidad (
    sk_colaborador  INTEGER NOT NULL REFERENCES dim_colaborador (sk_colaborador),
    sk_habilidad    INTEGER NOT NULL REFERENCES dim_habilidad (sk_habilidad),
    nivel           TEXT    NOT NULL CHECK (nivel IN ('Básico', 'Intermedio', 'Avanzado', 'Experto')),
    nivel_num       INTEGER NOT NULL CHECK (nivel_num BETWEEN 1 AND 4),
    PRIMARY KEY (sk_colaborador, sk_habilidad)
);

-- Índices para los filtros que más usa el tablero
CREATE INDEX ix_fact_colaborador ON fact_asignacion (sk_colaborador);
CREATE INDEX ix_fact_proyecto    ON fact_asignacion (sk_proyecto);
CREATE INDEX ix_fact_cliente     ON fact_asignacion (sk_cliente);
CREATE INDEX ix_fact_corte       ON fact_asignacion (sk_fecha_corte);
CREATE INDEX ix_puente_habilidad ON puente_colaborador_habilidad (sk_habilidad, nivel_num);
"""

with open('modelo_fisico.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(DDL)
print('modelo_fisico.sql guardado:', DDL.count('CREATE TABLE'), 'tablas y', DDL.count('CREATE INDEX'), 'índices')

modelo_fisico.sql guardado: 9 tablas y 5 índices


In [4]:
if os.path.exists('ot_estrella.db'):
    os.remove('ot_estrella.db')
con = sqlite3.connect('ot_estrella.db')
con.execute('PRAGMA foreign_keys = ON')    # SQLite solo revisa las llaves foráneas si se lo pides
con.executescript(DDL)

# primero las dimensiones y al final el hecho y el puente, para que las llaves foráneas ya existan
for nombre in ['dim_colaborador', 'dim_proyecto', 'dim_cliente', 'dim_fecha', 'dim_detalle_asignacion', 'dim_calidad',
               'dim_habilidad', 'fact_asignacion', 'puente_colaborador_habilidad']:
    t = TABLAS[nombre].astype(object).where(TABLAS[nombre].notna(), None)
    con.executemany(f"INSERT INTO {nombre} ({', '.join(t.columns)}) VALUES ({', '.join('?' * t.shape[1])})",
                    t.values.tolist())
con.commit()
sql = lambda consulta: pd.read_sql_query(consulta, con)

nombres = sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")['name']
pd.DataFrame([{'tabla': t, 'columnas': len(sql(f'PRAGMA table_info({t})')),
               'renglones': sql(f'SELECT COUNT(*) AS n FROM {t}')['n'][0],
               'llaves foráneas': len(sql(f'PRAGMA foreign_key_list({t})'))} for t in nombres])

,tabla,columnas,renglones,llaves foráneas
0,dim_calidad,3,18,0
1,dim_cliente,3,9,0
2,dim_colaborador,24,75,0
3,dim_detalle_asignacion,7,16,0
4,dim_fecha,8,1462,0
5,dim_habilidad,6,71,0
6,dim_proyecto,4,13,0
7,fact_asignacion,22,76,8
8,puente_colaborador_habilidad,4,1276,2


**¿Quedó bien?** Hacemos dos pruebas. La primera: que ninguna llave foránea apunte a algo que no existe. La segunda: volver a unir las tablas con SQL y comparar contra la tabla acondicionada, columna por columna. Si sale cero en las dos, no perdimos ni cambiamos nada al repartir.

In [5]:
print('Llaves foráneas rotas:', len(sql('PRAGMA foreign_key_check')))

reconstruida = sql('''
SELECT c.id_colaborador, c.nombre, c.tipo_colaborador, c.es_facturable, c.puesto, c.nivel, c.departamento, c.depto_origen,
       c.ciudad, c.nomina, c.antiguedad_anios, c.baja_programada, f.cuenta_persona, c.sueldo_total_integrado,
       c.costo_faltante, c.costo_mensual, c.costo_hora, c.tarifa_min_usd_20, c.tarifa_min_usd_30, c.n_habilidades,
       c.n_avanzado_experto, c.tecnologia_principal, c.captura_masiva, c.estado_banca, f.costo_banca_mes,
       NULLIF(p.codigo_proyecto, 'P000') AS codigo_proyecto, NULLIF(cl.codigo_cliente, 'C0000') AS codigo_cliente,
       NULLIF(cl.cliente, 'Sin cliente') AS cliente, NULLIF(p.proyecto, 'Sin asignación') AS proyecto,
       NULLIF(p.tipo_proyecto, 'No aplica') AS tipo_proyecto,
       NULLIF(d.rol_en_proyecto, 'No aplica') AS rol_en_proyecto, NULLIF(d.rol_origen, 'No aplica') AS rol_origen,
       fi.fecha AS inicio_asignacion, ff.fecha AS fin_asignacion, d.es_indefinida, d.asignacion_vigente,
       f.dias_para_liberarse, NULLIF(d.ventana_liberacion, 'No aplica') AS ventana_liberacion, f.dedicacion, f.horas_mes,
       f.costo_asignacion_mes, f.ingreso_mes_40usd, f.ingreso_mes_45usd, f.ingreso_mes_50usd,
       f.margen_40usd, f.margen_45usd, f.margen_50usd, d.fuera_de_periodo, f.puntaje_confianza, q.semaforo, q.campos_por_revisar
FROM fact_asignacion f
JOIN dim_colaborador c         USING (sk_colaborador)
JOIN dim_proyecto p            USING (sk_proyecto)
JOIN dim_cliente cl            USING (sk_cliente)
JOIN dim_fecha fi              ON fi.sk_fecha = f.sk_fecha_inicio
JOIN dim_fecha ff              ON ff.sk_fecha = f.sk_fecha_fin
JOIN dim_detalle_asignacion d  USING (sk_detalle_asignacion)
JOIN dim_calidad q             USING (sk_calidad)
ORDER BY f.id_hecho''')

original = tabla.reset_index(drop=True).copy()
for columna in ['inicio_asignacion', 'fin_asignacion']:
    original[columna] = pd.to_datetime(original[columna]).dt.strftime('%Y-%m-%d')
def iguales(x, y):
    try:
        return bool((((x.astype(float) - y.astype(float)).abs() < 1e-6) | (x.isna() & y.isna())).all())
    except (ValueError, TypeError):
        return bool((x.fillna('-').astype(str) == y.fillna('-').astype(str)).all())
distintas = [c for c in original.columns if not iguales(original[c], reconstruida[c])]
print(f'Reconstruida: {reconstruida.shape[0]} renglones x {reconstruida.shape[1]} columnas')
print('Columnas con diferencias contra la tabla acondicionada:', len(distintas), distintas)

Llaves foráneas rotas: 0
Reconstruida: 76 renglones x 51 columnas
Columnas con diferencias contra la tabla acondicionada: 0 []


## 4. ¿Qué valores tiene cada dimensión?

La regla del avance: si una columna tiene 10 valores distintos o menos, los listamos; si tiene más, la describimos (cuántos son, el rango y unos ejemplos). Para `dim_fecha` además sacamos el rango de fechas.

Los nombres y el esquema de nómina no los imprimimos. Aunque estos datos son de prueba, en la app real serían datos sensibles, así que solo damos cuántos hay.

In [6]:
SENSIBLES = {'nombre', 'nomina'}

def valores(tabla_sql):
    df = sql(f'SELECT * FROM {tabla_sql}')
    filas = []
    for columna in df.columns:
        if columna.startswith('sk_'):
            continue
        s = df[columna].dropna()
        n = s.nunique()
        entero = pd.api.types.is_numeric_dtype(s) and bool((s % 1 == 0).all())
        bonito = lambda v: f'{int(v)}' if entero else (f'{v:,.2f}' if isinstance(v, float) else str(v))
        if columna in SENSIBLES:
            detalle = f'{n} distintos (no se muestran por ser dato sensible)'
        elif n <= 10:
            detalle = ' | '.join(f'{bonito(v)} ({k})' for v, k in s.value_counts().items())
        elif pd.api.types.is_numeric_dtype(s):
            detalle = f'de {bonito(s.min())} a {bonito(s.max())}'
        else:
            detalle = f'{n} distintos, por ejemplo: ' + ', '.join(map(str, s.unique()[:3]))
        filas.append({'columna': columna, 'distintos': n, 'vacíos': int(df[columna].isna().sum()), 'valores': detalle})
    print(f'{tabla_sql}: {len(df)} renglones')
    return pd.DataFrame(filas)

with pd.option_context('display.max_colwidth', None):
    for d in ['dim_colaborador', 'dim_proyecto', 'dim_cliente', 'dim_detalle_asignacion', 'dim_calidad', 'dim_habilidad']:
        display(valores(d))

dim_colaborador: 75 renglones
                   columna  distintos  vacíos                                                                                                                       valores
0           id_colaborador         75       0                                                                             75 distintos, por ejemplo: OTC003, OTC004, OTC005
1                   nombre         75       0                                                                           75 distintos (no se muestran por ser dato sensible)
2         tipo_colaborador          6       0  Consultor (36) | Sin definir (15) | Consultor (inferido) (14) | Administrativo (6) | Gerente de proyectos (2) | Familiar (2)
3            es_facturable          2       0                                                                                                               1 (52) | 0 (23)
4                   puesto         23       0                23 distintos, por ejemplo: Sin definir, CONSULTOR

In [7]:
with pd.option_context('display.max_colwidth', None):
    display(valores('dim_fecha'))
sql('''
SELECT 'calendario completo' AS que, MIN(fecha) AS desde, MAX(fecha) AS hasta, COUNT(*) AS dias
FROM dim_fecha WHERE sk_fecha <> -1
UNION ALL
SELECT 'inicio de asignaciones', MIN(fecha), MAX(fecha), COUNT(DISTINCT fecha)
FROM dim_fecha WHERE sk_fecha IN (SELECT sk_fecha_inicio FROM fact_asignacion)
UNION ALL
SELECT 'fin de asignaciones', MIN(fecha), MAX(fecha), COUNT(DISTINCT fecha)
FROM dim_fecha WHERE sk_fecha IN (SELECT sk_fecha_fin FROM fact_asignacion)
UNION ALL
SELECT 'fecha de corte', MIN(fecha), MAX(fecha), COUNT(DISTINCT fecha)
FROM dim_fecha WHERE sk_fecha IN (SELECT sk_fecha_corte FROM fact_asignacion)''')

dim_fecha: 1462 renglones
      columna  distintos  vacíos                                                                                                                     valores
0       fecha       1461       1                                                             1461 distintos, por ejemplo: 2025-01-01, 2025-01-02, 2025-01-03
1        anio          4       1                                                                           2028 (366) | 2025 (365) | 2026 (365) | 2027 (365)
2   trimestre          4       1                                                                                       3 (368) | 4 (368) | 2 (364) | 1 (361)
3         mes         12       1                                                                                                                   de 1 a 12
4  nombre_mes         13       0                                                                        13 distintos, por ejemplo: Sin fecha, enero, febrero
5         dia         31       1

,que,desde,hasta,dias
0,calendario completo,2025-01-01,2028-12-31,1461
1,inicio de asignaciones,2025-11-03,2026-10-13,3
2,fin de asignaciones,2027-01-01,2028-07-01,3
3,fecha de corte,2026-11-16,2026-11-16,1


## 5. Las 12 preguntas

Cada pregunta es una consulta SQL sobre la estrella. Los ingresos y márgenes salen de los tres escenarios de tarifa de la etapa 3 (40, 45 y 50 USD/h), porque la app no guarda ni horas ni tarifa. Cuando la pregunta pide un solo número usamos 45 USD/h, que es el punto medio de lo que nos dijo el director. El margen se calcula sobre el costo, como lo mide OT.

### P1. ¿Cuántos consultores facturables tiene OT y cuánto cuesta cada uno por hora?

In [8]:
sql('''
SELECT tipo_colaborador,
       COUNT(*)                   AS personas,
       ROUND(AVG(costo_hora), 2)  AS costo_hora_promedio,
       ROUND(MIN(costo_hora), 2)  AS costo_hora_min,
       ROUND(MAX(costo_hora), 2)  AS costo_hora_max
FROM dim_colaborador
WHERE es_facturable = 1
GROUP BY tipo_colaborador
ORDER BY personas DESC''')

,tipo_colaborador,personas,costo_hora_promedio,costo_hora_min,costo_hora_max
0,Consultor,36,361.56,50.80,723.07
1,Consultor (inferido),14,487.52,231.72,884.25
2,Gerente de proyectos,2,513.93,339.68,688.18


Son **52 facturables**: 36 consultores, 14 que contamos como consultores aunque su puesto diga PENDIENTE (porque están en un proyecto) y 2 gerentes. Todos tienen costo. Un consultor cuesta en promedio **$362 por hora**, pero va de $51 a $723, así que no conviene cotizar a todos con la misma tarifa. El costo por hora de cada quien está en el tablero.

### P2. ¿Qué habilidades importantes dependen de una sola persona?

In [9]:
sql('''
SELECT h.tecnologia, h.nombre_completo AS habilidad, c.id_colaborador AS unica_persona, c.puesto, c.estado_banca, p.nivel
FROM puente_colaborador_habilidad p
JOIN dim_habilidad   h USING (sk_habilidad)
JOIN dim_colaborador c USING (sk_colaborador)
WHERE p.nivel_num >= 3                                   -- Avanzado o Experto
  AND p.sk_habilidad IN (SELECT sk_habilidad FROM puente_colaborador_habilidad
                         WHERE nivel_num >= 3 GROUP BY sk_habilidad HAVING COUNT(*) = 1)
ORDER BY h.tecnologia, habilidad''')

,tecnologia,habilidad,unica_persona,puesto,estado_banca,nivel
0,Oracle Fusion Cloud,Landed Cost · Oracle Fusion Cloud,OTC078,CONSULTOR FUNCIONAL ORACLE CLOUD SENIOR,Asignado,Avanzado
1,Oracle Fusion Cloud,MFG · Oracle Fusion Cloud,OTC094,CONSULTOR ORACLE CLOUD Y EBS SENIOR,En banca,Avanzado
2,Oracle eBS,XTR · Oracle eBS,OTC026,CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,En banca,Experto
3,Oracle eBS,eBTAX · Oracle eBS,OTC026,CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,En banca,Avanzado
4,Plataformas Tecnológicas,OCI · Plataformas Tecnológicas,OTC071,CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,En banca,Avanzado


In [10]:
sql('''
SELECT tecnologia, nombre_completo AS habilidad_sin_nadie_avanzado
FROM dim_habilidad
WHERE sk_habilidad NOT IN (SELECT sk_habilidad FROM puente_colaborador_habilidad WHERE nivel_num >= 3)
ORDER BY tecnologia''')

,tecnologia,habilidad_sin_nadie_avanzado
0,Oracle eBS,OPM · Oracle eBS
1,Oracle eBS,TALEO · Oracle eBS
2,Otros,Empatia · Otros
3,Otros,Manejo de Conflictos · Otros
4,Plataformas Tecnológicas,OBI · Plataformas Tecnológicas
5,Plataformas Tecnológicas,FUSE · Plataformas Tecnológicas


**5 habilidades** en nivel Avanzado o Experto las tiene una sola persona. Son 4 personas, porque OTC026 tiene dos de ellas, y 3 de las 4 están en la banca. Si alguna se va, OT se queda sin esa habilidad. Además hay **6 habilidades del catálogo que nadie tiene** en Avanzado o Experto.

### P3. ¿Quién está disponible con una habilidad y nivel específicos?

In [11]:
TECNOLOGIA, NIVEL_MINIMO = 'Oracle Fusion Cloud', 3      # cámbialos: 3 = Avanzado, 4 = Experto
sql(f'''
SELECT c.id_colaborador, c.puesto, c.estado_banca,
       COUNT(*)                            AS habilidades,
       GROUP_CONCAT(h.descripcion, ', ')   AS cuales,
       ROUND(c.costo_hora, 2)              AS costo_hora
FROM puente_colaborador_habilidad p
JOIN dim_habilidad   h USING (sk_habilidad)
JOIN dim_colaborador c USING (sk_colaborador)
WHERE h.tecnologia = '{TECNOLOGIA}'
  AND p.nivel_num >= {NIVEL_MINIMO}
  AND c.estado_banca IN ('En banca', 'Por confirmar')
GROUP BY c.sk_colaborador
ORDER BY habilidades DESC, costo_hora''')

,id_colaborador,puesto,estado_banca,habilidades,cuales,costo_hora
0,OTC046,Sin definir,Por confirmar,10,"GL, SLA, FA, TAX, PO, AP, AR, CM, BC, Expenses",356.62
1,OTC026,CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,En banca,9,"GL, SLA, CE, FA, PO, AP, AR, OM, INV",290.74
2,OTC071,CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,En banca,9,"GL, SLA, CE, FA, TAX, PO, AP, AR, INV",559.60
3,OTC016,CONSULTOR NETSUITE ORACLE SENIOR,En banca,8,"FA, TAX, PO, AP, AME, AR, OM, Expenses",348.52
4,OTC096,Sin definir,Por confirmar,8,"GL, TAX, PO, AP, AR, CM, BC, Expenses",393.45
5,OTC015,Sin definir,Por confirmar,6,"GL, FA, PO, AP, AR, INV",311.88
6,OTC094,CONSULTOR ORACLE CLOUD Y EBS SENIOR,En banca,6,"CE, FA, PO, AME, INV, MFG",361.36
7,OTC024,CONSULTOR FUNCIONAL ORACLE CLOUD Y EBS SENIOR,En banca,5,"PO, AR, OM, INV, WMS",312.48
8,OTC017,CONSULTOR TÉCNICO ORACLE EBS SENIOR,En banca,4,"GL, CE, AP, AR",325.17
9,OTC085,Sin definir,Por confirmar,1,AP,247.43


Como ejemplo buscamos Oracle Fusion Cloud en nivel Avanzado o más: hay **10 personas disponibles**, 6 en banca y 4 por confirmar. La consulta tiene la tecnología y el nivel como parámetros, y en el tablero se eligen con un menú.

### P4. ¿Cuántos consultores están en la banca y cuánto cuestan al mes?

In [12]:
sql('''
SELECT c.estado_banca, COUNT(*) AS personas, ROUND(SUM(c.costo_mensual), 0) AS costo_mensual
FROM fact_asignacion f
JOIN dim_colaborador c USING (sk_colaborador)
WHERE f.cuenta_persona = 1                               -- cada persona una sola vez
GROUP BY c.estado_banca
ORDER BY personas DESC''')

,estado_banca,personas,costo_mensual
0,Asignado,30,2309653.0
1,En banca,22,1363346.0
2,Por confirmar,15,829044.0
3,No facturable,8,200595.0


**22 consultores en banca cuestan $1,363,346 al mes**, igual que en la etapa 4. Si los 15 «por confirmar» también estuvieran sin proyecto, serían $2.19 millones. Esos 15 no tienen puesto ni proyecto en la app, y solo OT puede decir qué hacen.

### P5. ¿Quiénes se liberan en 30, 60 y 90 días?

In [13]:
sql('''
SELECT d.ventana_liberacion, COUNT(*) AS asignaciones,
       GROUP_CONCAT(c.id_colaborador || COALESCE(' (' || ff.fecha || ')', ''), ', ') AS quien_y_cuando
FROM fact_asignacion f
JOIN dim_detalle_asignacion d USING (sk_detalle_asignacion)
JOIN dim_colaborador c       USING (sk_colaborador)
JOIN dim_fecha ff            ON ff.sk_fecha = f.sk_fecha_fin
WHERE f.sk_proyecto <> 0
GROUP BY d.ventana_liberacion
ORDER BY MIN(COALESCE(f.dias_para_liberarse, 99999))''')

,ventana_liberacion,asignaciones,quien_y_cuando
0,31 a 60 días,3,"OTC014 (2027-01-01), OTC050 (2027-01-01), OTC0..."
1,Más de 90 días,4,"OTC104 (2027-07-01), OTC105 (2027-07-01), OTC1..."
2,Indefinida,18,"OTC004, OTC005, OTC009, OTC022, OTC031, OTC042..."
3,Sin fecha de fin,6,"OTC006, OTC056, OTC058, OTC079, OTC087, OTC100"


**Nadie se libera en los próximos 30 días.** 3 personas terminan el 1 de enero de 2027 (entre 31 y 60 días) y 4 en más de 90 días. Ojo: 24 asignaciones no tienen fecha de fin, 18 porque son indefinidas y 6 porque nadie la capturó. Para esas 6 no se puede planear.

### P6. ¿Qué porcentaje de la nómina está asignado a proyectos?

In [14]:
sql('''
SELECT ROUND(SUM(f.costo_asignacion_mes), 0)                          AS costo_asignado,
       ROUND((SELECT SUM(costo_mensual) FROM dim_colaborador), 0)     AS nomina_total,
       ROUND(100.0 * SUM(f.costo_asignacion_mes)
             / (SELECT SUM(costo_mensual) FROM dim_colaborador), 1)   AS pct_asignado
FROM fact_asignacion f
WHERE f.sk_proyecto <> 0''')

,costo_asignado,nomina_total,pct_asignado
0,2249093.0,4702637.0,47.8


**El 47.8% de la nómina** ($2.25 de $4.70 millones al mes) está en un proyecto. El resto es la banca, los por confirmar, los no facturables y la parte del tiempo de los gerentes que no está en proyectos.

### P7. ¿Hay personas con más del 100% de dedicación?

In [15]:
sql('''
SELECT c.id_colaborador, c.tipo_colaborador, COUNT(*) AS proyectos, SUM(f.dedicacion) AS dedicacion_total
FROM fact_asignacion f
JOIN dim_colaborador c USING (sk_colaborador)
WHERE f.sk_proyecto <> 0
GROUP BY c.sk_colaborador
HAVING COUNT(*) > 1 OR SUM(f.dedicacion) > 1''')

,id_colaborador,tipo_colaborador,proyectos,dedicacion_total
0,OTC095,Gerente de proyectos,2,0.5


**Nadie pasa del 100%.** El único que está en dos proyectos es un gerente (OTC095), con 25% en cada uno. Esta respuesta es débil porque la dedicación es estimada con la regla de OT: la app no la guarda. Si OT la empieza a capturar, la misma consulta da la respuesta real.

### P8. ¿Qué margen deja cada proyecto y cuáles quedan por debajo del 20%?

In [16]:
sql('''
SELECT p.codigo_proyecto, p.proyecto, COUNT(*) AS personas,
       ROUND(SUM(f.costo_asignacion_mes), 0) AS costo_mes,
       ROUND((SUM(f.ingreso_mes_40usd) - SUM(f.costo_asignacion_mes)) / SUM(f.costo_asignacion_mes), 3) AS margen_40,
       ROUND((SUM(f.ingreso_mes_45usd) - SUM(f.costo_asignacion_mes)) / SUM(f.costo_asignacion_mes), 3) AS margen_45,
       ROUND((SUM(f.ingreso_mes_50usd) - SUM(f.costo_asignacion_mes)) / SUM(f.costo_asignacion_mes), 3) AS margen_50,
       SUM(CASE WHEN f.margen_45usd < 0.20 THEN 1 ELSE 0 END) AS personas_bajo_20_a_45usd
FROM fact_asignacion f
JOIN dim_proyecto p USING (sk_proyecto)
WHERE f.sk_proyecto <> 0 AND f.costo_asignacion_mes IS NOT NULL
GROUP BY p.sk_proyecto
ORDER BY margen_45''')

,codigo_proyecto,proyecto,personas,costo_mes,margen_40,margen_45,margen_50,personas_bajo_20_a_45usd
0,P003,Asignación indefinida,18,1492356.0,0.509,0.698,0.887,4
1,P004,Marazzi,3,176579.0,0.664,0.872,1.080,0
2,P002,Soporte AMS Manufacta,5,382242.0,0.708,0.922,1.135,0
3,P007,Migración eBs,5,197915.0,2.299,2.712,3.124,0


**Ningún proyecto queda abajo del 20%**, ni siquiera cobrando 40 USD/h. El más bajo es «Asignación indefinida» de METALSA (51% a 40 USD/h), que además junta 18 personas y $1.49 millones de costo al mes. Ahí hay 4 personas que por sí solas quedan debajo del 20% a 45 USD/h, porque su costo por hora es alto. 8 de los 12 proyectos no tienen a nadie asignado.

### P9. ¿Qué clientes o tipos de proyecto generan mejores márgenes?

In [17]:
sql('''
SELECT 'cliente' AS vista, cl.cliente AS nombre,
       ROUND(SUM(f.ingreso_mes_45usd) - SUM(f.costo_asignacion_mes), 0) AS utilidad_mes_45,
       ROUND((SUM(f.ingreso_mes_45usd) - SUM(f.costo_asignacion_mes)) / SUM(f.costo_asignacion_mes), 3) AS margen_45
FROM fact_asignacion f JOIN dim_cliente cl USING (sk_cliente)
WHERE f.sk_cliente <> 0 AND f.costo_asignacion_mes IS NOT NULL
GROUP BY cl.sk_cliente
UNION ALL
SELECT 'tipo de proyecto', p.tipo_proyecto,
       ROUND(SUM(f.ingreso_mes_45usd) - SUM(f.costo_asignacion_mes), 0),
       ROUND((SUM(f.ingreso_mes_45usd) - SUM(f.costo_asignacion_mes)) / SUM(f.costo_asignacion_mes), 3)
FROM fact_asignacion f JOIN dim_proyecto p USING (sk_proyecto)
WHERE f.sk_proyecto <> 0 AND f.costo_asignacion_mes IS NOT NULL
GROUP BY p.tipo_proyecto
ORDER BY vista, margen_45 DESC''')

,vista,nombre,utilidad_mes_45,margen_45
0,cliente,Allegion,536665.0,2.712
1,cliente,Grupo Daltile,153982.0,0.872
2,cliente,METALSA,1394282.0,0.744
3,tipo de proyecto,Implementación,690647.0,1.844
4,tipo de proyecto,Póliza de soporte,352338.0,0.922
5,tipo de proyecto,Asignación de recursos,1041945.0,0.698


En porcentaje gana **Allegion** (271% a 45 USD/h) y, por tipo, **Implementación** (184%). Pero en dinero quien más deja es **METALSA: $1.39 millones al mes**, porque ahí está casi toda la gente. Por eso en el tablero ponemos las dos cosas, margen y utilidad.

### P10. ¿Cuáles son los 10 consultores que generan mayor utilidad?

In [18]:
sql('''
SELECT c.id_colaborador, c.puesto, p.proyecto, ROUND(c.costo_hora, 2) AS costo_hora,
       ROUND(f.ingreso_mes_45usd - f.costo_asignacion_mes, 0) AS utilidad_mes_45, f.margen_45usd AS margen_45
FROM fact_asignacion f
JOIN dim_colaborador c USING (sk_colaborador)
JOIN dim_proyecto p    USING (sk_proyecto)
WHERE f.sk_proyecto <> 0 AND f.costo_asignacion_mes IS NOT NULL
ORDER BY utilidad_mes_45 DESC
LIMIT 10''')

,id_colaborador,puesto,proyecto,costo_hora,utilidad_mes_45,margen_45
0,OTC107,CONSULTOR FUNCIONAL ORACLE EBS SENIOR,Migración eBs,95.62,130087.0,7.730
1,OTC104,CONSULTOR FUNCIONAL ORACLE EBS SENIOR,Migración eBs,204.22,110974.0,3.088
2,OTC105,CONSULTOR FUNCIONAL ORACLE EBS SENIOR,Migración eBs,204.22,110974.0,3.088
3,OTC106,CONSULTOR FUNCIONAL ORACLE EBS SENIOR,Migración eBs,204.22,110974.0,3.088
4,OTC031,Sin definir,Asignación indefinida,231.72,106133.0,2.602
5,OTC102,Sin definir,Asignación indefinida,231.72,106133.0,2.602
6,OTC050,CONSULTOR FUNCIONAL ORACLE CLOUD Y EBS ANALISTA,Marazzi,248.21,103231.0,2.363
7,OTC042,CONSULTOR KRONOS SENIOR,Asignación indefinida,259.64,101219.0,2.215
8,OTC060,Sin definir,Asignación indefinida,308.44,92631.0,1.706
9,OTC081,CONSULTOR FUNCIONAL ORACLE CLOUD SENIOR,Asignación indefinida,351.63,85029.0,1.374


Encabezan los 4 consultores de «Migración eBs» (Allegion), porque su costo por hora es de los más bajos. Como todos se cobran igual en el escenario, el ranking en realidad ordena por costo. Con la tarifa real de cada proyecto este top podría cambiar.

### P11. ¿Cuánto se espera facturar en los próximos meses?

In [19]:
sql('''
WITH meses(mes, inicio_mes) AS (
    VALUES ('dic 2026', 20261201), ('ene 2027', 20270101), ('feb 2027', 20270201), ('mar 2027', 20270301)
)
SELECT m.mes, COUNT(f.id_hecho) AS asignaciones_activas,
       ROUND(SUM(f.ingreso_mes_40usd), 0) AS factura_40usd,
       ROUND(SUM(f.ingreso_mes_45usd), 0) AS factura_45usd,
       ROUND(SUM(f.ingreso_mes_50usd), 0) AS factura_50usd
FROM meses m
LEFT JOIN fact_asignacion f
       ON f.sk_proyecto <> 0
      AND f.sk_fecha_inicio < m.inicio_mes + 100                  -- ya empezó (o no tiene fecha de inicio = -1)
      AND (f.sk_fecha_fin = -1 OR f.sk_fecha_fin > m.inicio_mes)  -- sigue después del día 1 o es indefinida
GROUP BY m.mes
ORDER BY m.inicio_mes''')

,mes,asignaciones_activas,factura_40usd,factura_45usd,factura_50usd
0,dic 2026,31,3852464.0,4334022.0,4815580.0
1,ene 2027,28,3558632.0,4003461.0,4448290.0
2,feb 2027,28,3558632.0,4003461.0,4448290.0
3,mar 2027,28,3558632.0,4003461.0,4448290.0


Con las asignaciones de hoy, en **diciembre se facturarían unos $4.33 millones** a 45 USD/h (entre $3.85 y $4.82 millones según la tarifa). En enero baja a $4.00 millones porque terminan 3 asignaciones. Es una estimación: supone 176 horas al mes por persona, porque la app no tiene lo vendido ni las horas reales.

### P12. ¿Qué tan completos están los registros y quién los mantiene actualizados?

In [20]:
display(sql('''
SELECT q.semaforo, COUNT(*) AS registros, ROUND(AVG(f.puntaje_confianza), 2) AS confianza_promedio,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fact_asignacion), 1) AS pct
FROM fact_asignacion f
JOIN dim_calidad q USING (sk_calidad)
GROUP BY q.semaforo
ORDER BY CASE q.semaforo WHEN 'Verde' THEN 1 WHEN 'Amarillo' THEN 2 ELSE 3 END'''))

sql('''
WITH RECURSIVE partes(sk_calidad, campo, resto) AS (       -- separa "rol, puesto" en renglones
    SELECT sk_calidad, '', campos_por_revisar || ', ' FROM dim_calidad WHERE campos_por_revisar IS NOT NULL
    UNION ALL
    SELECT sk_calidad, substr(resto, 1, instr(resto, ', ') - 1), substr(resto, instr(resto, ', ') + 2)
    FROM partes WHERE resto <> ''
)
SELECT p.campo AS campo_que_falta, COUNT(*) AS registros
FROM partes p JOIN fact_asignacion f USING (sk_calidad)
WHERE p.campo <> ''
GROUP BY p.campo
ORDER BY registros DESC''')

   semaforo  registros  confianza_promedio   pct
0     Verde         15                0.98  19.7
1  Amarillo         51                0.63  67.1
2      Rojo         10                0.28  13.2


,campo_que_falta,registros
0,rol,29
1,puesto,29
2,habilidades,28
3,fecha de inicio,21
4,fecha de fin,6
5,costo,5


Solo **el 20% de los registros está en verde**; 67% está en amarillo y 13% en rojo. Lo que más falta es el rol (29), el puesto (29) y las habilidades (28). ¿Quién los mantiene? **Eso no se puede saber con la app**: no guarda quién creó o cambió un registro, y en el sandbox los 6 usuarios son Administrador. Recomendamos agregar `modificado_por` y `fecha_modificacion` a cada colección.

## 6. Archivo para el tablero

El tablero de Dirección es una página web que lee un solo archivo, `datos_tablero.json`. Aquí lo armamos con consultas a la estrella. También guardamos la base `ot_estrella.db` y las 9 tablas en CSV, por si alguien las quiere abrir en Tableau o Excel.

In [21]:
def datos_tablero(con, pro, mostrar_nombres=True):
    # Todo lo que necesita la página de Dirección, sacado de la estrella con SQL
    q = lambda consulta: pd.read_sql_query(consulta, con)
    registros = lambda df: json.loads(df.to_json(orient='records', force_ascii=False, double_precision=4))

    personas = q('''
        SELECT c.id_colaborador AS id, c.nombre, c.tipo_colaborador AS tipo, c.es_facturable AS facturable, c.puesto,
               c.nivel, c.departamento AS depto, c.ciudad, c.antiguedad_anios AS antig,
               ROUND(c.costo_mensual, 2) AS costo_mes, ROUND(c.costo_hora, 2) AS costo_hora,
               c.tarifa_min_usd_20 AS tarifa_min_20, c.estado_banca AS estado, c.tecnologia_principal AS tecnologia,
               c.n_habilidades AS n_hab, c.n_avanzado_experto AS n_avanzado, q.semaforo, q.campos_por_revisar AS faltan
        FROM dim_colaborador c
        JOIN fact_asignacion f USING (sk_colaborador)
        JOIN dim_calidad q     USING (sk_calidad)
        WHERE f.cuenta_persona = 1
        ORDER BY c.id_colaborador''')
    if not mostrar_nombres:
        personas['nombre'] = personas['id']

    asignaciones = q('''
        SELECT c.id_colaborador AS id, p.codigo_proyecto AS proyecto_cod, p.proyecto, p.tipo_proyecto, cl.cliente,
               d.rol_en_proyecto AS rol, fi.fecha AS inicio, ff.fecha AS fin, d.es_indefinida AS indefinida,
               d.ventana_liberacion AS ventana, f.dias_para_liberarse AS dias, f.dedicacion, f.horas_mes AS horas,
               ROUND(f.costo_asignacion_mes, 2) AS costo, ROUND(f.ingreso_mes_40usd, 2) AS ing40,
               ROUND(f.ingreso_mes_45usd, 2) AS ing45, ROUND(f.ingreso_mes_50usd, 2) AS ing50, q.semaforo
        FROM fact_asignacion f
        JOIN dim_colaborador c        USING (sk_colaborador)
        JOIN dim_proyecto p           USING (sk_proyecto)
        JOIN dim_cliente cl           USING (sk_cliente)
        JOIN dim_detalle_asignacion d USING (sk_detalle_asignacion)
        JOIN dim_calidad q            USING (sk_calidad)
        JOIN dim_fecha fi             ON fi.sk_fecha = f.sk_fecha_inicio
        JOIN dim_fecha ff             ON ff.sk_fecha = f.sk_fecha_fin
        WHERE f.sk_proyecto <> 0
        ORDER BY p.codigo_proyecto, c.id_colaborador''')

    calidad = q('''
        SELECT q.semaforo, q.campos_por_revisar, f.puntaje_confianza
        FROM fact_asignacion f JOIN dim_calidad q USING (sk_calidad)''')
    faltan = calidad['campos_por_revisar'].dropna().str.split(', ').explode().value_counts()

    habilidades = q('''SELECT sk_habilidad AS sk, codigo_habilidad AS codigo, descripcion, tecnologia, tipo
                       FROM dim_habilidad ORDER BY tecnologia, descripcion''')
    puente = q('''SELECT c.id_colaborador AS id, p.sk_habilidad AS sk, p.nivel_num AS nivel
                  FROM puente_colaborador_habilidad p JOIN dim_colaborador c USING (sk_colaborador)''')
    matriz = {i: g[['sk', 'nivel']].values.tolist() for i, g in puente.groupby('id')}

    proyectos = pro[['codigo_proyecto', 'nombre', 'cliente', 'tipo_proyecto', 'estatus']] \
        .rename(columns={'codigo_proyecto': 'codigo', 'tipo_proyecto': 'tipo'}).sort_values('codigo')

    return {
        'corte': str(FECHA_CORTE.date()), 'tipo_cambio': TIPO_CAMBIO, 'horas_mes': HORAS_MES,
        'carga_admin': CARGA_ADMIN, 'escenarios': TARIFAS_USD, 'meta_margen': list(META_MARGEN),
        'personas': registros(personas), 'asignaciones': registros(asignaciones),
        'proyectos': registros(proyectos), 'habilidades': registros(habilidades), 'matriz': matriz,
        'calidad': {'semaforo': calidad['semaforo'].value_counts().to_dict(),
                    'faltan': faltan.to_dict(), 'registros': len(calidad)},
    }

datos = datos_tablero(con, pro)
with open('datos_tablero.json', 'w', encoding='utf-8') as archivo:
    json.dump(datos, archivo, ensure_ascii=False, separators=(',', ':'))
os.makedirs('estrella_csv', exist_ok=True)
for nombre, t in TABLAS.items():
    t.to_csv(f'estrella_csv/{nombre}.csv', index=False, encoding='utf-8-sig')
print('datos_tablero.json:', f"{os.path.getsize('datos_tablero.json') / 1024:.0f} KB,",
      len(datos['personas']), 'personas,', len(datos['asignaciones']), 'asignaciones,', len(datos['habilidades']), 'habilidades')

datos_tablero.json: 57 KB, 75 personas, 31 asignaciones, 71 habilidades


In [22]:
# En Colab: descarga todo en un zip
if 'google.colab' in str(get_ipython()):
    with zipfile.ZipFile('etapa5_salida.zip', 'w') as z:
        for ruta in ['datos_tablero.json', 'ot_estrella.db', 'modelo_fisico.sql'] + \
                    [f'estrella_csv/{n}.csv' for n in TABLAS]:
            z.write(ruta)
    from google.colab import files
    files.download('etapa5_salida.zip')

## 7. ¿Cómo se repite esto cuando cambian los datos?

Nadie tiene que volver a abrir este notebook. En la carpeta `tablero` del repo está `actualizar_tablero.py`, que hace todo seguido:

1. **Extrae** las 9 colecciones de Firebase con una cuenta de servicio de solo lectura y las aplana en los 11 CSV. Los datos personales (RFC, CURP, NSS, correo y teléfono) se cambian por 1/0 desde ese momento.
2. **Limpia** con el notebook de la etapa 3.
3. **Arma la estrella** y la carga en SQL con las celdas de este notebook marcadas como `pipeline`.
4. **Escribe** `datos_tablero.json` junto a la página. OT lo publica en su mismo Firebase Hosting con `firebase deploy --only hosting`.

Si se programa cada lunes con Cloud Scheduler o GitHub Actions, el tablero se actualiza solo.

## 8. Lo que nos llevamos

- **El modelo aguanta.** Las 7 tablas reconstruyen la tabla acondicionada completa, sin diferencias, y ninguna llave foránea quedó rota. Las reglas `CHECK` evitan que vuelvan a entrar datos de relleno como los que encontramos en la etapa 2.
- **Las 12 preguntas se contestan con la estrella**, más un puente para habilidades. Las de banca, liberaciones, habilidades y calidad ya son respuestas firmes. Las de dinero (P8 a P11) siguen siendo escenarios, porque la app no tiene tarifa, horas ni monto vendido.
- **Lo que más le sirve a Dirección:** 22 consultores en banca cuestan $1.36 millones al mes, y hay 10 personas disponibles con Oracle Fusion Cloud avanzado. Antes de contratar a alguien conviene buscar en la banca.